# Kaggle Word2Vec NLP Tutorial Part 4：Comparing Deep and Non-Deep Learning Methods

Part 4 的重点不是再引入一个全新的复杂模型，而是比较前面几种方法。

核心问题是：为什么 Word2Vec 更接近深度学习思想，但在这个教程里可能没有明显超过 Bag-of-Words。


## 1. 导入库


In [ ]:
import csv
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

from gensim.models import Word2Vec
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split


## 2. 读取数据


In [ ]:
def show_available_inputs():
    input_root = Path('/kaggle/input')
    if input_root.exists():
        print('Available /kaggle/input entries:')
        for path in sorted(input_root.glob('*')):
            print(' -', path)
            for child in sorted(path.glob('*'))[:10]:
                print('    -', child.name)
    else:
        print('/kaggle/input does not exist. If running locally, put files under data/.')

def find_file(filename, search_dirs, required=True):
    for directory in search_dirs:
        directory = Path(directory)
        direct_path = directory / filename
        zipped_path = directory / f'{filename}.zip'
        if direct_path.exists():
            return direct_path
        if zipped_path.exists():
            return zipped_path
        if directory.exists():
            matches = list(directory.rglob(filename))
            if matches:
                return matches[0]
            zipped_matches = list(directory.rglob(f'{filename}.zip'))
            if zipped_matches:
                return zipped_matches[0]
    if required:
        show_available_inputs()
        raise FileNotFoundError(
            f'Cannot find {filename}. In Kaggle, click Add Input / Add data and attach the word2vec-nlp-tutorial competition dataset.'
        )
    return None

def read_tsv(path):
    path = Path(path)
    if path.suffix == '.zip':
        with zipfile.ZipFile(path) as archive:
            tsv_names = [name for name in archive.namelist() if name.endswith('.tsv')]
            if not tsv_names:
                raise ValueError(f'No TSV file found inside {path}')
            with archive.open(tsv_names[0]) as file_obj:
                return pd.read_csv(file_obj, header=0, delimiter='\t', quoting=csv.QUOTE_NONE)
    return pd.read_csv(path, header=0, delimiter='\t', quoting=csv.QUOTE_NONE)

search_dirs = [
    '/kaggle/input/word2vec-nlp-tutorial',
    '/kaggle/input',
    'data',
    '.',
]

train_path = find_file('labeledTrainData.tsv', search_dirs)
test_path = find_file('testData.tsv', search_dirs, required=False)
unlabeled_path = find_file('unlabeledTrainData.tsv', search_dirs)

print('Train file:    ', train_path)
print('Test file:     ', test_path)
print('Unlabeled file:', unlabeled_path)


train = read_tsv(train_path)
unlabeled_train = read_tsv(unlabeled_path)
test = read_tsv(test_path) if test_path is not None else None

print('Train shape:    ', train.shape)
print('Unlabeled shape:', unlabeled_train.shape)
if test is not None:
    print('Test shape:     ', test.shape)

display(train.head())


## 3. 公共清洗函数


In [ ]:
STOP_WORDS = set(ENGLISH_STOP_WORDS)

def review_to_wordlist(raw_review, remove_stopwords=False):
    text = BeautifulSoup(raw_review, 'html.parser').get_text()
    text = re.sub('[^a-zA-Z]', ' ', text)
    words = text.lower().split()
    if remove_stopwords:
        words = [word for word in words if word not in STOP_WORDS]
    return words

def simple_sentence_split(raw_review):
    return [sentence.strip() for sentence in re.split(r'[.!?]+', raw_review) if sentence.strip()]

try:
    import nltk.data
    tokenizer = nltk.data.load('tokenizers/punkt/english.pickle')
    print('Using NLTK punkt sentence tokenizer.')
except Exception:
    tokenizer = None
    print('NLTK punkt tokenizer is not available. Falling back to simple regex sentence splitting.')

def review_to_sentences(raw_review, remove_stopwords=False):
    if tokenizer is not None:
        raw_sentences = tokenizer.tokenize(raw_review.strip())
    else:
        raw_sentences = simple_sentence_split(raw_review)

    sentences = []
    for raw_sentence in raw_sentences:
        words = review_to_wordlist(raw_sentence, remove_stopwords=remove_stopwords)
        if words:
            sentences.append(words)
    return sentences


def review_to_words(raw_review):
    return ' '.join(review_to_wordlist(raw_review, remove_stopwords=True))


## 4. 切分本地验证集

Kaggle 的测试集没有标签，不能直接算准确率。

所以这里从 `labeledTrainData.tsv` 里划出 20% 当验证集，用来比较不同方法。


In [ ]:
train_text, valid_text, y_train, y_valid = train_test_split(
    train['review'],
    train['sentiment'].astype(int),
    test_size=0.2,
    random_state=42,
    stratify=train['sentiment'].astype(int),
)

print('Train split:', train_text.shape)
print('Valid split:', valid_text.shape)


## 5. 方法一：Bag-of-Words + RandomForest

这是 Part 1 的方法。它简单，但在情感分类里往往很强。


In [ ]:
clean_train_bow = [review_to_words(review) for review in train_text]
clean_valid_bow = [review_to_words(review) for review in valid_text]

vectorizer = CountVectorizer(max_features=5000)
X_train_bow = vectorizer.fit_transform(clean_train_bow)
X_valid_bow = vectorizer.transform(clean_valid_bow)

forest_bow = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
)
forest_bow.fit(X_train_bow, y_train)
pred_bow = forest_bow.predict(X_valid_bow)

bow_score = accuracy_score(y_valid, pred_bow)
print('Bag-of-Words validation accuracy:', bow_score)


## 6. 方法二：Average Word2Vec + RandomForest

这里先训练或加载 Word2Vec，然后把每篇评论表示成平均词向量。


In [ ]:
MODEL_PATHS = [
    Path('/kaggle/working/300features_40minwords_10context.model'),
    Path('300features_40minwords_10context.model'),
]
model_path = next((path for path in MODEL_PATHS if path.exists()), None)

if model_path is not None:
    print('Loading Word2Vec model:', model_path)
    model = Word2Vec.load(str(model_path))
else:
    print('No saved Word2Vec model found. Training Word2Vec for comparison...')
    sentences = []
    for review in train['review']:
        sentences.extend(review_to_sentences(review, remove_stopwords=False))
    for review in unlabeled_train['review']:
        sentences.extend(review_to_sentences(review, remove_stopwords=False))

    model = Word2Vec(
        sentences=sentences,
        vector_size=300,
        min_count=40,
        workers=4,
        window=10,
        sample=1e-3,
        sg=0,
        seed=42,
    )

NUM_FEATURES = model.wv.vector_size
print('Vocabulary size:', len(model.wv.index_to_key))


In [ ]:
def make_feature_vec(words, model, num_features):
    feature_vec = np.zeros((num_features,), dtype='float32')
    n_words = 0
    vocab = set(model.wv.index_to_key)

    for word in words:
        if word in vocab:
            feature_vec += model.wv[word]
            n_words += 1

    if n_words > 0:
        feature_vec /= n_words

    return feature_vec

def get_avg_feature_vecs(reviews, model, num_features):
    review_vecs = np.zeros((len(reviews), num_features), dtype='float32')
    for i, review in enumerate(reviews):
        if i % 1000 == 0:
            print(f'Review {i} of {len(reviews)}')
        review_vecs[i] = make_feature_vec(review, model, num_features)
    return review_vecs

clean_train_wordlist = [review_to_wordlist(review, remove_stopwords=True) for review in train_text]
clean_valid_wordlist = [review_to_wordlist(review, remove_stopwords=True) for review in valid_text]

X_train_avg = get_avg_feature_vecs(clean_train_wordlist, model, NUM_FEATURES)
X_valid_avg = get_avg_feature_vecs(clean_valid_wordlist, model, NUM_FEATURES)

forest_avg = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
)
forest_avg.fit(X_train_avg, y_train)
pred_avg = forest_avg.predict(X_valid_avg)

avg_score = accuracy_score(y_valid, pred_avg)
print('Average Word2Vec validation accuracy:', avg_score)


## 7. 汇总比较结果


In [ ]:
scores = pd.DataFrame({
    'method': [
        'Bag-of-Words + RandomForest',
        'Average Word2Vec + RandomForest',
    ],
    'validation_accuracy': [
        bow_score,
        avg_score,
    ],
})

scores = scores.sort_values('validation_accuracy', ascending=False)
display(scores)


## 8. 为什么 Bag-of-Words 可能更好

Word2Vec 更高级，但在这个教程里不一定直接超过 Bag-of-Words。

主要原因有几个。

第一，平均词向量会丢失词序。

`not good` 和 `good` 的差别会被削弱。

第二，平均词向量会稀释情感词。

一篇长评论里真正表达情感的词可能很少，平均之后信号会变弱。

第三，这里的 Word2Vec 训练语料不算特别大。

教程使用的是 75,000 条评论，大约千万级词量；Google 原始 Word2Vec 常用的是十亿级语料。

第四，随机森林对稠密连续向量不一定是最合适的模型。

所以 Part 4 想表达的不是 Word2Vec 没用，而是文本表示方法和下游模型要配合。更复杂的方法如果聚合方式太粗糙，也可能输给简单但直接的词袋模型。


## 9. 小结

Part 1 是传统词袋特征。

Part 2 是训练词向量。

Part 3 是把词向量变成评论特征。

Part 4 是比较这些方法，理解它们各自的信息损失。

这个教程的价值在于，它展示了早期 NLP 从词频模型走向 embedding 的过程。现代大模型的 embedding 更复杂，但“把文本变成向量再用于下游任务”这个思路是一脉相承的。
